# ML Laboratuvarı · Ünite 10: Bitirme projesi

**Prof. Dr. Mehmet Songur** · ML Laboratuvarı, Modül 0

Bir soruyu gerçek veriyle **Gemini'nin yardımıyla** yanıtla, her adımı **denetle** ve **belgele**.

| Soru | Seriler (FRED) | Odak |
|---|---|---|
| **A** | `CPIAUCNS`, `UNRATE` | Enflasyon ve işsizlik birlikte nasıl hareket etti? |
| **B** | `FEDFUNDS`, `CPIAUCNS` | Politika faizi ve enflasyon; reel faizin negatif olduğu dönemler |
| **C** | `GDPC1`, `UNRATE` | Çeyreklik reel büyüme ve işsizlikteki değişim |

Dönem: **2000–2024**. Veriyi indirmek için hazır `fred_indir("SERI_KODU")` fonksiyonunu kullan; Gemini'nin başka kaynak ya da seri kodu uydurmasına izin verme.

**Teslim:** `veri` tablosu, en az bir grafik, `yorum` paragrafı ve `denetim` sözlüğü. Son hücre bunları kontrol eder ve tamamlama kodunu verir.

**Son kontrolün eşikleri (önceden bil):**
- `veri`: A ve B için **300 aylık** gözlem (2000-01 … 2024-12), C için **100 çeyreklik** gözlem (2000Ç1 … 2024Ç4); tarihe göre artan sırada; sayısal sütunlarda eksik değer yok; en az iki sayısal sütun.
- `yorum`: en az **300 karakter**; sonucun sınırlarına (nedensellik, genelleme…) değinmeli.
- `denetim`: beş alanın **her biri en az 30 karakter**.

Son kontrol biçimi ve kapsamı sınar; grafiğin ve hesabın doğruluğunu, yorumun niteliğini sınamaz. Onlar için 5. adımdaki öz değerlendirme formunu kullan.

**Gemini ile çalışma:** önerilen kodu **Kabul et** ile hücreye al, çalıştırmadan önce oku ve gerekirse düzelt.

In [ ]:
#@title Kurulum — çalıştır, değiştirme

import io, math, base64, urllib.request
import pandas as pd

def fred_indir(seri_kodu):
    """FRED'den bir seriyi indirir; sütunlar: tarih, <seri_kodu>. FRED eksik değerleri '.' ile gösterir;
    bunlar bilerek NaN yapılır ve sayısı yazdırılır."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + seri_kodu
    with urllib.request.urlopen(url) as yanit:
        df = pd.read_csv(io.BytesIO(yanit.read()))
    df.columns = ["tarih", seri_kodu]
    df["tarih"] = pd.to_datetime(df["tarih"])
    df[seri_kodu] = pd.to_numeric(df[seri_kodu], errors="coerce")
    print(f"{seri_kodu}: {len(df)} satır, {df[seri_kodu].isna().sum()} eksik değer, {df['tarih'].min():%Y-%m} – {df['tarih'].max():%Y-%m}")
    return df

def _k(s):
    return base64.b64decode(s).decode()

_SINIR = ["nedensel", "neden-sonuç", "neden sonuç", "sınır", "genelle", "kısıt", "göstermez"]

def kontrol_bitirme(veri, yorum, denetim):
    sorun = []
    soru = str(globals().get("SORU", "A")).upper()
    frek = "Q" if soru == "C" else "M"
    n_bek = 100 if frek == "Q" else 300
    if not isinstance(veri, pd.DataFrame):
        sorun.append("'veri' bir pandas tablosu olmalı.")
    else:
        t = pd.to_datetime(veri["tarih"]) if "tarih" in veri.columns else pd.to_datetime(pd.Series(veri.index))
        if not t.is_monotonic_increasing:
            sorun.append("'veri' tarihe göre artan sırada değil.")
        if t.min() < pd.Timestamp("2000-01-01") or t.max() > pd.Timestamp("2024-12-31"):
            sorun.append("Dönem 2000–2024 ile sınırlı değil.")
        donem = pd.PeriodIndex(t.dt.to_period(frek))
        beklenen = pd.period_range("2000-01" if frek == "M" else "2000Q1", "2024-12" if frek == "M" else "2024Q4", freq=frek)
        if len(veri) != n_bek or not donem.equals(beklenen):
            ad = "aylık" if frek == "M" else "çeyreklik"
            sorun.append(f"Soru {soru} için {n_bek} {ad} gözlem (2000–2024, boşluksuz ve tekrarsız) bekleniyor; tabloda {len(veri)} satır var. Frekansı kontrol et; gecikmeli dönüşümleri (yıllık değişim, büyüme) 2000'e kesmeden önce mi yaptın?")
        if veri.select_dtypes("number").shape[1] < 2:
            sorun.append("'veri' en az iki sayısal sütun içermeli (sorunun iki değişkeni).")
        if veri.select_dtypes("number").isna().any().any():
            sorun.append("Sayısal sütunlarda eksik değer var; nedenini anla ve açıkça ele al.")
    if not isinstance(yorum, str) or len(yorum.strip()) < 300:
        sorun.append("'yorum' en az 300 karakter olmalı.")
    elif not any(k in yorum.lower() for k in _SINIR):
        sorun.append("Yorumda sonucun sınırlarına (nedensellik, genelleme…) değinilmemiş.")
    beklenen = ["veri", "istekler", "kontroller", "hatalar", "sinirlar"]
    if not isinstance(denetim, dict) or any(len(str(denetim.get(k, "")).strip()) < 30 for k in beklenen):
        sorun.append("'denetim' sözlüğünün beş alanının her biri en az 30 karakter olmalı.")
    if sorun:
        print("❌ Henüz değil:")
        for s in sorun:
            print("  •", s)
        return
    print("✅ Tüm kontroller geçti. Tamamlama kodun: " + _k("TUwwLUJUUi05SERa"))
    print("Not: Bu kontrol biçimi denetler; analizin doğruluğundan ve yorumun dürüstlüğünden sen sorumlusun.")

In [ ]:
#@title Sorunu seç
SORU = "A"  #@param ["A", "B", "C"]
print("Seçilen soru:", SORU)

## 1. Veri

Gemini'den sırasıyla şunları iste:
1. Seçtiğin sorunun iki serisini `fred_indir` ile indir; **1999 verisini de tut**.
2. Serileri **ortak frekansa** getir (C için aylıktan çeyreğe) ve tarihe göre birleştir.
3. Gecikme gerektiren dönüşümleri **bu aşamada** yap: yıllık enflasyon (12 ay öncesine göre), C için çeyreklik büyüme ve işsizlikteki değişim. 1999 verisi sayesinde 2000'in ilk ayları da hesaplanabilir.
4. **Ancak ondan sonra** 2000–2024 dönemine kes. Sonuç tablonun adı `veri` olsun ve `tarih` sütunu bulunsun.

Önce 2000–2024'e kesip sonra 12 aylık değişim hesaplarsan 2000'in ilk 12 ayı boş kalır.

**Denetim:** satır sayısı (A ve B: 300, C: 100), ilk ve son tarih, eksik değerler, frekans, birimler (yüzde mi düzey mi?).

**Aylık takvim kontrolü:** Bir `Series` ile bir `PeriodIndex`'i doğrudan `==` ile karşılaştırma; uzunluklar farklıysa `ValueError` verir. İkisini aynı türe getir:
```python
donem = pd.PeriodIndex(pd.to_datetime(veri["tarih"]).dt.to_period("M"))
beklenen = pd.period_range("2000-01", "2024-12", freq="M")   # C için: freq="Q", "2000Q1", "2024Q4"
print(donem.equals(beklenen))   # True olmalı; uzunluk farklıysa hata değil False verir
```

In [ ]:
# Gemini ile kodu burada üret, oku, sonra çalıştır.

## 2. Analiz ve grafik

Sorunun gerektirdiği dönüşümleri (yıllık enflasyon, reel faiz, büyüme oranı, işsizlikteki değişim…) Gemini'ye yaptır, en az bir **özet sayı** (ör. korelasyon) ve bir **grafik** üret.

**Denetim:** Birim hatası? Sıralama? Geometrik/aritmetik ortalama? Grafiğin başlığı ve eksen adları var mı?

In [ ]:
# Gemini ile kodu burada üret, oku, sonra çalıştır.

## 3. Yorum

Dört parçalı bir paragraf yaz: **bulgu, büyüklük, bağlam, sınırlar**. Gemini'ye yazdırırsan Ünite 8'deki gibi denetle. Nedensellik iddia etme.

In [ ]:
yorum = """
Buraya en az 4 cümlelik yorumunu yaz.
"""

## 4. Denetim

Her alana en az bir iki cümle yaz. Bu alanları ML Laboratuvarı'ndaki denetim defterine de aktaracaksın.

In [ ]:
denetim = {
    "veri": "",        # Hangi seriler, dönem, frekans? Satır sayısı ve eksikler?
    "istekler": "",    # Gemini'ye verdiğin en önemli istekler ve neden öyle yazdığın
    "kontroller": "",  # Uyguladığın doğrulama soruları ve sonuçları
    "hatalar": "",     # Bulduğun hatalar ve düzeltmeler (ya da "bulmadım, çünkü...")
    "sinirlar": "",    # Sonucun neyi göstermediği
}

### 5. Son kontrol — çalıştır, değiştirme

Kontrol biçimi ve kapsamı sınar (eşikler en üstte). Çalıştırmadan önce şu **öz değerlendirme formunu** kendin doldur; bunları otomatik kontrol göremez:

| Ölçüt | Evet / Hayır |
|---|---|
| Grafiğin başlığı, eksen adları ve birimleri var; doğru serileri gösteriyor | |
| Özet sayıyı (ör. korelasyon) en az bir alt dönemde ya da elle yeniden hesaplayıp doğruladım | |
| Dönüşümlerin birimi doğru (yüzde, yüzde puan, düzey) | |
| Yorum dört parçalı: bulgu, büyüklük, bağlam, sınırlar | |
| Yorumda nedensellik iddiası yok; açıklamalar hipotez olarak yazıldı | |

In [ ]:
kontrol_bitirme(veri, yorum, denetim)